In [1]:
!pip install transformers accelerate bitsandbytes sentence-transformers faiss-cpu -q
# transformers: loads the Qwen language model and tokenizer
# accelerate: handles multi-GPU / automatic device placement
# bitsandbytes: enables 4-bit quantization to reduce VRAM usage
# sentence-transformers: loads the multilingual embedding model for retrieval
# faiss-cpu: vector similarity search library (CPU version)
# -q: quiet mode, suppresses verbose install output
print("Fertig!") # Confirm all packages installed successfully

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.6 MB/s eta 0:00:00:00:0100:01
Fertig!


In [2]:
import json                                        # For reading the chunks JSON file
import pandas as pd                                # For loading and working with the test CSV dataset
import numpy as np                                 # Numerical operations (used later with vectors)
import torch                                       # PyTorch — the deep learning framework for model inference

with open("/kaggle/input/datasets/anso1234/alle-chunks/alle_chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)                          # Load the pre-prepared law text chunks from disk into a Python list
print(f"Chunks geladen: {len(chunks)}")            # Print how many chunks are available (should be ~6685)

dataset_url = "https://raw.githubusercontent.com/svakulenk0/wu-llms-ss26/main/dataset_clean.csv"  # URL of the test question dataset
df = pd.read_csv(dataset_url)                      # Download and parse the CSV directly from GitHub into a DataFrame
print(f"Dataset geladen: {len(df)} Fragen")        # Print how many questions the test set contains
print(f"Spalten: {df.columns.tolist()}")           # Print the column names of the DataFrame
print(f"\nErste Zeile:")                           # Label for the row preview
print(df.iloc[0])                                  # Print the first row to inspect its structure

Chunks geladen: 6685
Dataset geladen: 643 Fragen
Spalten: ['id', 'prompt']

Erste Zeile:
id                                             CORP-TAX-001
prompt    Was ist die steuerliche Bemessungsgrundlage fü...
Name: 0, dtype: object


In [3]:
from sentence_transformers import SentenceTransformer  # Library for loading pre-trained embedding models
import faiss                                           # Facebook's vector similarity search library

print("Lade Retriever...")                         # Announce model download/loading
retriever = SentenceTransformer("intfloat/multilingual-e5-base")
                                                   # Load the multilingual E5 embedding model — handles German text well
print("Retriever geladen")                         # Confirm the model is ready

print("\nWandle Chunks in Vektoren um...")         # Announce the embedding step
chunk_texte = [c['text'] for c in chunks]          # Extract just the text strings from the chunk dicts into a plain list

chunk_vektoren = retriever.encode(                 # Convert all chunk texts into dense vector representations
    chunk_texte,
    show_progress_bar=True,                        # Show a progress bar since this takes a while with 6685 chunks
    batch_size=64,                                 # Process 64 chunks at a time for memory efficiency
    convert_to_numpy=True                          # Return numpy arrays (required by FAISS)
)

print(f"Vektoren erstellt: {chunk_vektoren.shape}")  # Print the shape, e.g. (6685, 768) — 6685 chunks, 768 dimensions each

dimension = chunk_vektoren.shape[1]                # Extract the vector dimension (768 for E5-base)
faiss.normalize_L2(chunk_vektoren)                 # Normalize all vectors to unit length — required for cosine similarity via inner product

index = faiss.IndexFlatIP(dimension)               # Create a FAISS flat index using Inner Product (= cosine similarity after normalization)
index.add(chunk_vektoren)                          # Add all chunk vectors to the index

print(f"FAISS Index aufgebaut mit {index.ntotal} Vektoren")  # Confirm how many vectors are in the searchable index

Lade Retriever...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Retriever geladen

Wandle Chunks in Vektoren um...


Batches:   0%|          | 0/105 [00:00<?, ?it/s]

Vektoren erstellt: (6685, 768)
FAISS Index aufgebaut mit 6685 Vektoren


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
                                                   # AutoTokenizer: loads the tokenizer matching the model
                                                   # AutoModelForCausalLM: loads the text-generation model
                                                   # BitsAndBytesConfig: configures 4-bit quantization

model_name = "Qwen/Qwen2.5-3B-Instruct"           # Use Alibaba's Qwen 2.5 3B instruction-tuned model

quantization_config = BitsAndBytesConfig(          # Define 4-bit quantization settings to reduce VRAM usage
    load_in_4bit=True,                             # Load model weights in 4-bit precision
    bnb_4bit_quant_type="nf4",                     # Use NormalFloat4 quantization type — better quality than plain int4
    bnb_4bit_compute_dtype=torch.float16,          # Perform actual computations in float16 for speed
    bnb_4bit_use_double_quant=True                 # Apply double quantization to further compress the quantization constants
)

print("Lade Tokenizer...")                         # Announce tokenizer loading
tokenizer = AutoTokenizer.from_pretrained(model_name)  # Download and load the tokenizer for Qwen 2.5

print("Lade Modell auf GPU...")                    # Announce model loading
model = AutoModelForCausalLM.from_pretrained(      # Download and load the model weights
    model_name,
    quantization_config=quantization_config,       # Apply the 4-bit quantization defined above
    device_map="auto",                             # Automatically distribute the model across available GPUs
    trust_remote_code=True                         # Allow Qwen's custom model code to run (required by this model)
)
model.eval()                                       # Switch model to evaluation mode — disables dropout and gradient tracking

print("Modell geladen")                            # Confirm the model is ready
print(f"GPU Speicher verwendet: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")  # Print how much GPU memory is currently occupied

Lade Tokenizer...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modell geladen
GPU Speicher verwendet: 3.2 GB


In [7]:
import re                                          # Regular expressions for cleaning answer text
import time                                        # For measuring elapsed time and computing ETAs

def retrieve_chunks(frage, top_k=5):               # Function: find the most relevant law chunks for a given question
    frage_vektor = retriever.encode(               # Embed the question into a vector
        ["query: " + frage],                       # Prepend "query: " — required by the E5 model for query-mode encoding
        convert_to_numpy=True                      # Return as numpy array for FAISS
    )
    faiss.normalize_L2(frage_vektor)               # Normalize the query vector to unit length (same as the chunk vectors)
    scores, indices = index.search(frage_vektor, top_k)  # Search the FAISS index for the top_k most similar chunks
    return [chunks[idx] for idx in indices[0] if idx != -1]  # Return the matching chunk dicts; skip -1 (no match found)


def generate_answer(frage, relevante_chunks):      # Function: generate a German answer using the retrieved chunks as context
    kontext_teile = []                             # List to build the context string piece by piece
    for chunk in relevante_chunks:                 # Loop through each retrieved chunk
        kontext_teile.append(f"[{chunk['quelle']}]\n{chunk['text']}")  # Format each chunk as "[EStG 1988 § 23]\n<text>"
    kontext = "\n\n".join(kontext_teile)           # Join all formatted chunks into one context block

    messages = [                                   # Build the chat message list in the format Qwen expects
        {
            "role": "system",                      # System message: defines the model's role and constraints
            "content": (
                "Du bist ein Experte für österreichisches Steuerrecht. "  # Role definition
                "Beantworte die Frage ausschließlich auf Deutsch und ausschließlich "  # Answer must be in German only
                "auf Basis der bereitgestellten Gesetzestexte. "          # Answer must be based solely on the provided law texts
                "Antworte präzise und nenne den relevanten Paragraphen."  # Be concise and cite the relevant paragraph
            )
        },
        {
            "role": "user",                        # User message: contains the retrieved context and the question
            "content": f"Gesetzliche Grundlagen:\n{kontext}\n\nFrage: {frage}"
                                                   # Structure: law text chunks first, then the actual question
        }
    ]

    # apply_chat_template nur zum Text generieren, dann separat tokenisieren
    text = tokenizer.apply_chat_template(          # Convert the messages list into a single formatted string
        messages,
        tokenize=False,                            # Return a plain string, not token IDs yet
        add_generation_prompt=True                 # Append the assistant turn start marker so the model knows to generate
    )

    # Tokenisieren mit truncation separat
    inputs = tokenizer(                            # Now tokenize the formatted string
        text,
        return_tensors="pt",                       # Return PyTorch tensors
        truncation=True,                           # Truncate if the combined prompt exceeds max_length
        max_length=2048                            # Maximum input length in tokens
    ).to("cuda")                                   # Move input tensors to the GPU

    with torch.no_grad():                          # Disable gradient tracking — not needed during inference, saves memory
        outputs = model.generate(                  # Run the model to generate an answer
            **inputs,                              # Pass the tokenized input
            max_new_tokens=150,                    # Generate at most 150 new tokens for the answer
            do_sample=False,                       # Use greedy decoding — always pick the most likely token (deterministic)
            pad_token_id=tokenizer.eos_token_id    # Use the EOS token as the padding token to suppress warnings
        )

    neue_tokens = outputs[0][inputs['input_ids'].shape[-1]:]  # Slice off the prompt tokens — keep only the newly generated tokens
    antwort = tokenizer.decode(neue_tokens, skip_special_tokens=True)  # Decode token IDs back into readable text
    return antwort.strip()                         # Remove leading/trailing whitespace and return the answer


def clean_answer(antwort):                         # Function: normalize whitespace in the generated answer
    antwort = antwort.replace("\n", " ")           # Replace all newlines with spaces (answers should be single-line)
    antwort = re.sub(r'\s+', ' ', antwort)         # Collapse multiple consecutive spaces into one
    return antwort.strip()                         # Remove any remaining leading/trailing whitespace


prompt_spalte = None                               # Initialize the column name variable as empty
for col in df.columns:                             # Loop through all column names in the test dataset
    if col.lower() in ['prompt', 'question', 'frage', 'text']:  # Check if this column likely contains the questions
        prompt_spalte = col                        # Store the matching column name
        break                                      # Stop searching once found
if prompt_spalte is None:                          # If none of the expected column names were found...
    prompt_spalte = df.columns[1]                  # Fall back to the second column (index 1), assuming column 0 is "id"

print(f"Prompt-Spalte: '{prompt_spalte}'")         # Print which column will be used as the question input
print(f"Beispiel: {df[prompt_spalte].iloc[0][:100]}")  # Print the first 100 characters of the first question as a sanity check

Prompt-Spalte: 'prompt'
Beispiel: Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?


In [8]:
print(f"Starte RAG Pipeline für {len(df)} Fragen...")  # Announce the start of the full evaluation run

ergebnisse = []                                    # List to collect all generated answers
fehler_count = 0                                   # Counter for questions that caused errors
start_time = time.time()                           # Record the start time for ETA calculations

for i, row in df.iterrows():                       # Iterate over every row (question) in the test dataset

    frage_id   = row['id']                         # Extract the question's ID for the output CSV
    frage_text = row[prompt_spalte]                # Extract the question text from the identified column

    try:                                           # Wrap in try/except so one bad question doesn't crash the whole run
        relevante_chunks = retrieve_chunks(frage_text, top_k=5)  # Step 1: retrieve the 5 most relevant law chunks
        antwort = generate_answer(frage_text, relevante_chunks)   # Step 2: generate an answer using the model + context
        antwort_sauber = clean_answer(antwort)                    # Step 3: clean up newlines and extra whitespace

    except Exception as e:                         # If anything goes wrong for this question...
        fehler_count += 1                          # Increment the error counter
        print(f"  Fehler bei {frage_id}: {e}")    # Print the error message and the question ID
        antwort_sauber = "Fehler bei der Verarbeitung"  # Store a placeholder instead of crashing

    ergebnisse.append({"id": frage_id, "answer": antwort_sauber})  # Append the result (always, even on error)

    if (i + 1) % 5 == 0:                          # Every 5 questions, print a progress update
        elapsed      = time.time() - start_time   # Total seconds elapsed since the start
        done         = i + 1                       # Number of questions completed so far
        remaining    = len(df) - done              # Number of questions still to process
        per_question = elapsed / done              # Average time per question in seconds
        eta_seconds  = remaining * per_question    # Estimated seconds until completion
        elapsed_str  = time.strftime("%H:%M:%S", time.gmtime(elapsed))    # Format elapsed time as HH:MM:SS
        eta_str      = time.strftime("%H:%M:%S", time.gmtime(eta_seconds))  # Format ETA as HH:MM:SS
        print(f"[{done}/{len(df)}] | bisherige Zeit: {elapsed_str} | noch ca.: {eta_str} | Fehler: {fehler_count}")

        # Zwischenspeichern damit bei Timeout nichts verloren geht
        pd.DataFrame(ergebnisse).to_csv(           # Save intermediate results after every 5 questions
            "/kaggle/working/rag_qwen_ergebnisse.csv",  # Kaggle working directory — survives session
            index=False, sep=",", encoding="utf-8"
        )

ergebnisse_df = pd.DataFrame(ergebnisse)          # Convert the final results list into a DataFrame
output_path = "/kaggle/working/rag_qwen_ergebnisse.csv"  # Final output file path
ergebnisse_df.to_csv(output_path, index=False, sep=",", encoding="utf-8")
                                                   # Save the complete results as a CSV (overwriting the last intermediate save)

print(f"\nFertig!")                                # Announce completion
print(f"  {len(ergebnisse_df)} Antworten gespeichert")   # Print total number of answers written
print(f"  {fehler_count} Fehler aufgetreten")             # Print how many questions caused errors
print(f"  Gespeichert: {output_path}")             # Print the final file path

print("\n=== Erste 5 Ergebnisse ===")              # Section header for the preview
print(ergebnisse_df.head(5).to_string())           # Print the first 5 rows of the results DataFrame as a table

Starte RAG Pipeline für 643 Fragen...
[5/643] | bisherige Zeit: 00:01:18 | noch ca.: 02:47:53 | Fehler: 0
[10/643] | bisherige Zeit: 00:02:27 | noch ca.: 02:35:35 | Fehler: 0
[15/643] | bisherige Zeit: 00:03:45 | noch ca.: 02:37:20 | Fehler: 0
[20/643] | bisherige Zeit: 00:05:04 | noch ca.: 02:37:57 | Fehler: 0
[25/643] | bisherige Zeit: 00:06:24 | noch ca.: 02:38:20 | Fehler: 0
[30/643] | bisherige Zeit: 00:07:34 | noch ca.: 02:34:37 | Fehler: 0
[35/643] | bisherige Zeit: 00:08:52 | noch ca.: 02:34:17 | Fehler: 0
[40/643] | bisherige Zeit: 00:10:12 | noch ca.: 02:33:47 | Fehler: 0
[45/643] | bisherige Zeit: 00:11:21 | noch ca.: 02:30:57 | Fehler: 0
[50/643] | bisherige Zeit: 00:12:38 | noch ca.: 02:29:50 | Fehler: 0
[55/643] | bisherige Zeit: 00:13:56 | noch ca.: 02:29:04 | Fehler: 0
[60/643] | bisherige Zeit: 00:15:16 | noch ca.: 02:28:21 | Fehler: 0
[65/643] | bisherige Zeit: 00:16:33 | noch ca.: 02:27:18 | Fehler: 0
[70/643] | bisherige Zeit: 00:17:52 | noch ca.: 02:26:19 | Fehler: